<a target="_blank" href="https://colab.research.google.com/github/univiemops/tewa1-computational-cognition/blob/main/extra/05%20Recap.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


Recap lab 5
===

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy import linalg

## Exercise 1

### A. Multiple predictors

Write a function `my_mult_regr()` to perform the above calculation. This function should take 3 inputs in the following order: 1. predictor (age), 2. predictor (size), 3. outcome variable (price).

Your function must
1. create a predictor matrix (as above), starting with a column of ones, and the two predictors. (3 columns in total)
2. use `lstsq ()` to fit the regression model, as above
3. the function should return 2 outputs, the first is an array containing the 3 fitted regression parameters (1st output argument of `lstsq()`), the second output should be the residual error (2nd output argument of lstsq()),

Make sure that your function works for inputs of any size (this is important when you add the column of ones), but you can assume that all 3 input vectors have the same length (otherwise the analysis makes no sense).





In [ ]:
def my_mult_regr(x_1, x_2, y):
    """Computes linear regression with 2 predictors.

    Parameters
    ----------
    x_1, x_2 : array-like
        Predictor variables.
    y : array-like
        Outcome variable.

    Returns
    -------
    array-like
        Regression coefficients.
    float
        Total error.
    """
    if len(x_1) == len(x_2) == len(y):
        X = np.column_stack((np.ones(len(x_1)), x_1, x_2))
        coeff, total_error, _, _ = linalg.lstsq(X, y)
        return coeff, total_error
    else:
        raise Exception("All arrays must have the same length!")


# create some sample data

n = 250
sd_prices = 10_000
b_0 = 30_000  # intercept; ficticious average price at age == 0 and m² == 0
b_1 = -1_000  # slope for age
b_2 = 2_000  # slope for m²
sq_mtrs = np.random.randint(20, 200, n)
ages = np.random.randint(0, 100, n)
errors = np.random.normal(0, sd_prices, n)
prices = b_0 + b_1 * ages + b_2 * sq_mtrs + errors

# set a minimum of prices (for the sake of realism)

min_limit = 10_000
prices = np.where(prices < min_limit, min_limit, prices)

# run function and show results

coeff, total_error = my_mult_regr(ages, sq_mtrs, prices)
print(
    f"Result:\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nb_2: {coeff[2]:.2f}\nTotal error: {total_error:.2e}"
)

In [ ]:
plt.scatter(ages, prices, s=sq_mtrs, alpha=0.4)
plt.xlabel("age (yrs)")
plt.ylabel("price")
plt.legend(["size (m²)"])
plt.title("apartment prices");

### B. Standardized predictors

- Standardize (z-score) your predictors by subtracting the mean and dividing by the standard deviation.

- Fit a regression with both the single predictor and the two predictor models, and compare the error and the coefficients for fitting the model to standardized and unstandardized data sets.

- Make use of the `my_mult_regr()` function.

In [ ]:
def get_z_scores(x):
    """Computes z-scores.

    Parameters
    ----------
    x : array-like
        An array like object containing the sample data.

    Returns
    -------
    zscores : array_like
        The z-scores, standardized by mean and standard deviation of
        input array `x`.
    """

    return (x - np.mean(x)) / np.std(x)


# standardize predictors

ages_z = get_z_scores(ages)
sq_mtrs_z = get_z_scores(sq_mtrs)

#### Both predictors

In [ ]:
coeff, total_error = my_mult_regr(ages, sq_mtrs, prices)
print(
    f"Previous result (no standardization):\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nb_2: {coeff[2]:.2f}\nTotal error: {total_error:.2e}"
)

coeff, total_error = my_mult_regr(ages_z, sq_mtrs_z, prices)
print(
    f" \nResult with standardized predictors:\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nb_2: {coeff[2]:.2f}\nTotal error: {total_error:.2e}"
)

#### Age as the only predictor

In [ ]:
coeff, total_error, _, _ = linalg.lstsq(ages[:, np.newaxis] ** [0, 1], prices)
print(
    f" \nResult without standardized predictors:\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nTotal error: {total_error:.2e}"
)

coeff, total_error, _, _ = linalg.lstsq(ages_z[:, np.newaxis] ** [0, 1], prices)
print(
    f" \nResult with standardized predictors:\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nTotal error: {total_error:.2e}"
)

#### Size as the only predictor

In [ ]:
coeff, total_error, _, _ = linalg.lstsq(sq_mtrs[:, np.newaxis] ** [0, 1], prices)
print(
    f" \nResult without standardized predictors:\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nTotal error: {total_error:.2e}"
)

coeff, total_error, _, _ = linalg.lstsq(sq_mtrs_z[:, np.newaxis] ** [0, 1], prices)
print(
    f" \nResult with standardized predictors:\n\nb_0: {coeff[0]:.2f}\nb_1: {coeff[1]:.2f}\nTotal error: {total_error:.2e}"
)

## Exercise 2: Car price simulation

A new car costs an average of 30,000 Euros. Simulate 200 car prices from the last 70 years, assuming that while cars get cheaper as they get older, very old cars have a vintage value, that is, if they are old enough, they could eventually be worth more than a new car. Use a standard deviation of 10,000 Euros. Hint: Use a linear model for the simulation with two predictors and one linear and one quadratic term. Test different values for the two slopes, simulating data until you are able to simulate realistic car prices that meet the above criteria.

In [ ]:
mean_price_new = 30_000
sd_price = 10_000
n = 200
age_lower = 0
age_upper = 70

b_0 = mean_price_new  # avg value at age = 0
b_1 = -1500  # value loss per year
b_2 = 22  # value gain per year*year

Once you have found good values for this simulation, make a nice visualization of the simulated data.

In [ ]:
# creating ages array (uniformly distributed)
ages = np.sort(np.random.randint(age_lower, age_upper + 1, n))

# error array
err = np.random.normal(0, scale=sd_price, size=n)

# prices arrays
prices = np.array([b_0 + b_1 * ages[i] + b_2 * ages[i] ** 2 + err[i] for i in range(n)])
price_func = b_0 + b_1 * ages + b_2 * ages**2

# limit lowest price to 500
prices = np.where(prices < 500, 500, prices)

plt.plot(ages, price_func)
plt.scatter(ages, prices, alpha=0.4)
plt.ylabel("price")
plt.xlabel("age (yrs)")
plt.title("car price simulation");

Once, the data simulation is ready, fit 3 regression models to the simulated data:
1. intercept + linear predictor $age$
2. intercept + linear predictor $age$ + quadratic predictor $age$<sup>2</sup>
3. intercept + linear predictor $age$ + quadratic predictor $age$<sup>2</sup> + cubic predicor $age$<sup>3</sup>

`print()` the obtained residual error for the three models and visualize the model predictions

### 1. intercept + linear predictor $age$

- using `linalg.lstsq()`

In [ ]:
from scipy import linalg

# adding required column of ones to the predictor variable (because of matrix- vector multiplication)
ages_ = np.stack([ages, np.ones(n)], axis=1)

coeff, total_error, _, _ = linalg.lstsq(ages_, prices)
price_pred_1 = np.array([coeff[1] + coeff[0] * ages[i] for i in range(n)])

print("total error (scipy.linalg):", format(total_error, "e"))
print("intercept b_0 =", coeff[1])
print("slope b_1 =", coeff[0])

In [ ]:
res = (price_pred_1 - prices) ** 2
total_error_1 = np.sum(res)
print("total error (manually calculated):", format(total_error_1, "e"))

In [ ]:
polyline = np.linspace(age_lower, age_upper)
price_func = b_0 + b_1 * polyline + b_2 * polyline**2

plt.scatter(ages, prices, alpha=0.4)
plt.plot(ages, price_pred_1, c="red", label="regression line")
plt.plot(polyline, price_func, c="black", label="original line", linestyle=":")
plt.ylabel("price")
plt.xlabel("age (yrs)")
plt.legend()
plt.title("car price simulation: prediction 1");

### 2. intercept + linear predictor $age$ + quadratic predictor $age$²

- using `np.polyfit()`

In [ ]:
model_2 = np.poly1d(np.polyfit(ages, prices, 2))

price_reg_2 = model_2(polyline)

plt.scatter(ages, prices, alpha=0.4)
plt.plot(polyline, price_reg_2, c="red", label="regression line")
plt.plot(polyline, price_func, c="black", label="original line", linestyle=":")
plt.legend()
plt.title("car price simulation: prediction 2")
plt.show()

In [ ]:
print("intercept b_0 =", model_2[0])
print("slope b_1 =", model_2[1])
print("slope b_2 =", model_2[2])

In [ ]:
price_pred_2 = model_2(ages)
res_2 = (price_pred_2 - prices) ** 2
total_error_2 = np.sum(res_2)
print("total error (manually calculated):", format(total_error_2, "e"))

### 3. intercept + linear predictor $age$ + quadratic predictor $age$² + cubic predicor $age$³

- using `np.polyfit()`

In [ ]:
model_3 = np.poly1d(np.polyfit(ages, prices, 3))

price_reg_3 = model_3(polyline)

plt.scatter(ages, prices, alpha=0.4)
plt.plot(polyline, price_reg_3, c="red", label="regression line")
plt.plot(polyline, price_func, c="black", label="original line", linestyle=":")
plt.legend()
plt.title("car price simulation: prediction 3")
plt.show()

In [ ]:
print("intercept b_0 =", model_3[0])
print("slope b_1 =", model_3[1])
print("slope b_2 =", model_3[2])
print("slope b_3 =", model_3[3])

In [ ]:
price_pred_3 = model_3(ages)
res_3 = (price_pred_3 - prices) ** 2
total_error_3 = np.sum(res_3)
print("total error (manually calculated):", format(total_error_3, "e"))

## Bonus task: Reliability of regression analysis

Since we created the data, we can see how close are the true values to the 'generative' model. Next task is to systematically investigate this relationship. You will have to manipulate the number of datapoints, and the error in the model, and analyze the difference between the data generating and the fitted regression parameters. This task is somewhat analogous to the *t*-test simulation task

In [ ]:
def sim_error(n_points, sd, n_sims=10):
    """
    function for simulating total errors between regression line and original line
    of the quadratic function (we omit the other models).

    Parameters
    ----------
    n_points : int
        Number of data points.
    sd : float
        Standard deviation.
    n_sim int
        Number of simulations (Default = 10)

    Returns
    -------
    list-like
        List of the total mean errors of all simulations.
    """
    age_lower = 0  # fixed values
    age_upper = 70

    total_errors = list()

    for i in range(n_sims):

        ages_sim = np.sort(np.random.randint(age_lower, age_upper + 1, n_points))
        err_sim = np.random.normal(0, scale=sd, size=n_points)
        prices_sim = np.array(
            [
                b_0 + b_1 * ages_sim[i] + b_2 * ages_sim[i] ** 2 + err_sim[i]
                for i in range(n_points)
            ]
        )
        prices_sim = np.where(prices_sim < 0, 0, prices_sim)

        polyline_sim = np.linspace(age_lower, age_upper)
        price_func_sim = b_0 + b_1 * polyline_sim + b_2 * polyline_sim**2

        model_sim = np.poly1d(np.polyfit(ages_sim, prices_sim, 2))
        price_reg_sim = model_sim(polyline_sim)

        res_sim = (price_reg_sim - price_func_sim) ** 2
        total_error_sim = np.sum(res_sim)
        total_errors.append(total_error_sim)

    total_error_mean = np.mean(total_errors)

    return total_error_mean


n_space = range(50, 600, 50)
sd_space = range(100, 30_000, 4_000)

n_cases_sim = n_space[0]
sd_sim = sd_space[0]

results_arr = np.zeros((len(sd_space), len(n_space)))

print("Number of permutations:", np.size(results_arr))

In [ ]:
%%time

n_sim = 20  # would be better with a higher number, but it takes very long

for i_sd, sd in enumerate(sd_space):
    for i_n, n in enumerate(n_space):
        results_arr[i_sd, i_n] = sim_error(n, sd, n_sim)

plt.pcolor(results_arr)
plt.xticks(np.arange(len(n_space)), n_space)
plt.xlabel("n datapoints", fontsize=14)
plt.yticks(np.arange(len(sd_space)), sd_space, fontsize=14)
plt.ylabel("sd")

plt.colorbar()
plt.title("total errors original vs regression line");

### 2 Examples

In [ ]:
def sim_curves(n_points, sd):
    """
    Function for plots and text output for a specific constellation
    of number of data points and standard deviation. Only for demonstrating some examples.

    Parameters
    ----------
    n_points : int
        Number of data points.
    sd : float
        Standard deviation.

    Returns
    -------
    None.
    """

    age_lower = 0  # hardcoded values
    age_upper = 70  # ...

    ages_sim = np.random.randint(age_lower, age_upper + 1, n_points)
    err_sim = np.random.normal(0, scale=sd, size=n_points)
    prices_sim = np.array(
        [
            b_0 + b_1 * ages_sim[i] + b_2 * ages_sim[i] ** 2 + err_sim[i]
            for i in range(n_points)
        ]
    )
    prices_sim = np.where(prices_sim < 0, 0, prices_sim)

    polyline_sim = np.linspace(age_lower, age_upper)
    price_func_sim = b_0 + b_1 * polyline_sim + b_2 * polyline_sim**2

    model_sim = np.poly1d(np.polyfit(ages_sim, prices_sim, 2))
    price_reg_sim = model_sim(polyline_sim)

    res_sim = (price_reg_sim - price_func_sim) ** 2
    total_error_sim = np.sum(res_sim)

    plt.scatter(ages_sim, prices_sim, alpha=0.4)
    plt.plot(polyline_sim, price_reg_sim, c="red", label="regression line")
    plt.plot(
        polyline_sim, price_func_sim, c="black", label="original line", linestyle=":"
    )
    plt.legend()
    plt.title("car price simulation: regression 2")
    plt.show()

    print("total error:", format(total_error_sim, "e"))

    return None

#### High error: few points, high standard deviation

In [ ]:
n_points = 50
sd = 25_000

sim_curves(n_points, sd)

#### Low error: many points, low standard deviation

In [ ]:
n_points = 200
sd = 5_000

sim_curves(n_points, sd)

In [ ]:
# error measure could be improved ... is now absolute squared error (not really comparable),
# could be turned into MSE or RMSE for better comparability